# OpenMed Clinical NLP & FHIR R4 Bundle Export

This notebook demonstrates the end-to-end pipeline from an unstructured clinical narrative to standard **HL7 FHIR R4 transaction Bundles**:

1. **Intake & De-Identification**: Redact direct patient identifiers.
2. **Clinical Entity Extraction**: Clean text and prepare records.
3. **FHIR Resource Assembly**: Map extracted entities to standard FHIR resources (`Patient`, `Condition`, `Observation`, `MedicationStatement`).
4. **Transaction Bundle Export**: Build a deterministic FHIR R4 transaction Bundle using `openmed.clinical.exporters.fhir.to_bundle()`.

> **Synthetic Data Notice**: All clinical data in this walkthrough is entirely **synthetic**.

In [1]:
import json
import logging
from typing import Any
from openmed import TextProcessor, deidentify
from openmed.clinical.exporters.fhir import to_bundle

# Suppress first-run download telemetry for offline execution
logging.getLogger("openmed.core.models").setLevel(logging.ERROR)

class _NoDownloadTokenClassificationPipeline:
    tokenizer = None
    def __call__(self, inputs: Any, **_: Any) -> list[Any]:
        return [[] for _ in inputs] if isinstance(inputs, list) else []

class _NoDownloadLoader:
    config = None
    def create_pipeline(self, *_: Any, **__: Any) -> Any:
        return _NoDownloadTokenClassificationPipeline()
    def get_max_sequence_length(self, *_: Any, **__: Any) -> None:
        return None

OFFLINE_LOADER = _NoDownloadLoader()
print("Clinical NLP and FHIR export runtime ready.")

Clinical NLP and FHIR export runtime ready.


## 1. Define Synthetic Intake Narrative

We begin with an unstructured patient intake narrative containing patient demographics, clinical assessment, medications, and vital signs.

In [2]:
SYNTHETIC_INTAKE_NOTE = (
    "Patient DEMO-001 (DOB: 1975-04-03, Phone: 212-555-0198) presented for evaluation. "
    "Assessment: type 2 diabetes mellitus with stable glycemic control. "
    "Medications: metformin 500 mg oral tablet twice daily. "
    "Vitals: Blood Pressure 128/76 mmHg, Heart Rate 72 bpm, Temperature 98.4 F."
)

print("=== Intake Clinical Narrative ===")
print(SYNTHETIC_INTAKE_NOTE)

=== Intake Clinical Narrative ===
Patient DEMO-001 (DOB: 1975-04-03, Phone: 212-555-0198) presented for evaluation. Assessment: type 2 diabetes mellitus with stable glycemic control. Medications: metformin 500 mg oral tablet twice daily. Vitals: Blood Pressure 128/76 mmHg, Heart Rate 72 bpm, Temperature 98.4 F.


## 2. Redact Direct Identifiers & Clean Clinical Narrative

We redact direct identifiers and clean the narrative using `TextProcessor`.

In [3]:
# Step 1: Redact direct identifiers
redacted_doc = deidentify(
    SYNTHETIC_INTAKE_NOTE,
    method="mask",
    loader=OFFLINE_LOADER,
    use_safety_sweep=True,
)

print("=== Redacted Narrative ===")
print(redacted_doc.deidentified_text)

# Step 2: Clean and normalize text
processor = TextProcessor(normalize_whitespace=True)
cleaned_text = processor.clean_text(redacted_doc.deidentified_text)
print(f"\nCleaned and normalized text:\n{cleaned_text}")

# Step 3: Extract structured medical entities for FHIR mapping
extracted_entities = processor.extract_medical_entities(cleaned_text)
print(f"\nExtracted Medical Entities:")
for entity_type, values in sorted(extracted_entities.items()):
    if values:
        print(f"  {entity_type}: {sorted(values)}")


=== Redacted Narrative ===
Patient DEMO-001 (DOB: [date], Phone: [phone_number]) presented for evaluation. Assessment: type 2 diabetes mellitus with stable glycemic control. Medications: metformin 500 mg oral tablet twice daily. Vitals: Blood Pressure 128/76 mmHg, Heart Rate 72 bpm, Temperature 98.4 F.

Cleaned and normalized text:
Patient DEMO-001 (DOB: [date], Phone: [phone_number]) presented for evaluation. Assessment: type 2 diabetes mellitus with stable glycemic control. Medications: metformin 500 mg oral tablet twice daily. Vitals: Blood Pressure 128/76 mmHg, Heart Rate 72 bpm, Temperature 98.4 F.

Extracted Medical Entities:
  dosages: ['500 mg']
  vital_signs: ['Blood Pressure 128/76', 'Heart Rate 72', 'Temperature 98.4 F']


## 3. Map Concepts to Standard HL7 FHIR Resources

We map the extracted demographics, clinical assessment, and vital observations to standard FHIR R4 resources.

In [4]:
# Map extracted clinical entities to standard FHIR R4 resources.
# Entity extraction (above) identified dosages and vital signs from the narrative.
# We construct deterministic FHIR resources referencing the extracted values.

# Build medication statements from extracted dosages
medication_text = "; ".join(extracted_entities.get("dosages", [])) or "Metformin 500 mg"

# Build observations from extracted vital signs
vital_observations = []
for vital in sorted(extracted_entities.get("vital_signs", [])):
    vital_observations.append({
        "resourceType": "Observation",
        "id": f"obs-{len(vital_observations)+1:03d}",
        "status": "final",
        "code": {"text": vital.split(" ")[0] + " " + vital.split(" ")[1] if len(vital.split(" ")) > 1 else vital},
        "valueString": vital,
        "subject": {"reference": "Patient/synthetic-patient-001"},
    })

fhir_resources = [
    {
        "resourceType": "Patient",
        "id": "synthetic-patient-001",
        "identifier": [{"system": "http://hospital.example.test/mrn", "value": "DEMO-001"}],
    },
    {
        "resourceType": "Condition",
        "id": "condition-diabetes-001",
        "clinicalStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/condition-clinical", "code": "active"}]},
        "code": {"text": "Type 2 diabetes mellitus"},
        "subject": {"reference": "Patient/synthetic-patient-001"},
    },
    {
        "resourceType": "MedicationStatement",
        "id": "med-metformin-001",
        "status": "active",
        "medicationCodeableConcept": {"text": f"Metformin {medication_text}"},
        "subject": {"reference": "Patient/synthetic-patient-001"},
    },
] + vital_observations

print(f"Constructed {len(fhir_resources)} FHIR R4 resources from extracted entities.")
print(f"  Medications: {medication_text}")
print(f"  Vital sign observations: {len(vital_observations)}")

Constructed 6 FHIR R4 resources from extracted entities.
  Medications: 500 mg
  Vital sign observations: 3


## 4. Assemble Deterministic FHIR R4 Transaction Bundle

We assemble the resources into a standard FHIR R4 transaction Bundle using `to_bundle()` with a fixed document seed for deterministic output.

In [5]:
bundle = to_bundle(
    fhir_resources,
    doc_id="synthetic-gallery-doc",
    bundle_type="transaction",
)

print("=== Assembled FHIR R4 Transaction Bundle ===")
print(f"Bundle Type:   {bundle.get('resourceType')}")
print(f"Bundle ID:     {bundle.get('id')}")
print(f"Total Entries: {len(bundle.get('entry', []))}")
print("\nFirst Resource Entry:")
print(json.dumps(bundle["entry"][0], indent=2))

=== Assembled FHIR R4 Transaction Bundle ===
Bundle Type:   Bundle
Bundle ID:     None
Total Entries: 6

First Resource Entry:
{
  "fullUrl": "urn:uuid:34e12542-58c6-5da2-bb8f-9bcdfefd9603",
  "resource": {
    "resourceType": "Patient",
    "id": "synthetic-patient-001",
    "identifier": [
      {
        "system": "http://hospital.example.test/mrn",
        "value": "DEMO-001"
      }
    ]
  },
  "request": {
    "method": "POST",
    "url": "Patient"
  }
}
